In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [2]:
import jax.numpy as jnp

from hcnn.solver.solver_osqp import (
    OSQPParameters,
    OSQPProblem,
    osqp_solve,
)

A = jnp.array([[1.0, 0.0], [0.0, 1.0], [1.0, 0.0]])


def matrix_vector_product(x):
    return A @ x


def vector_matrix_product(y):
    return A.T @ y


osqp_problem = OSQPProblem(
    x0=jnp.array([0.0, 0.0]),
    cost_quad_diag=jnp.array([1.0, 1.0]),
    cost_linear=jnp.array([0.0, 0.0]),
    lower_bound=jnp.array([0.5, 0.5, 0.0]),
    upper_bound=jnp.array([1.0, 1.0, 0.6]),
    matrix_vector_product=matrix_vector_product,
    vector_matrix_product=vector_matrix_product,
)

osqp_variables = osqp_solve(
    problem=osqp_problem,
    parameters=OSQPParameters(),
)

print(osqp_variables)

OSQPVariables(x_tilde=Array([0.50000316, 0.50000435], dtype=float32), z_tilde=Array([0.50000316, 0.50000435, 0.50000316], dtype=float32), x=Array([0.500003  , 0.50000405], dtype=float32), z=Array([0.5     , 0.5     , 0.500003], dtype=float32), y=Array([-0.49999803, -0.49999732,  0.        ], dtype=float32), primal_residuals=Array([2.9802322e-06, 4.0531158e-06, 0.0000000e+00], dtype=float32), dual_residuals=Array([4.9471855e-06, 6.7353249e-06], dtype=float32), rho_base_mult=Array(2., dtype=float32), iteration=Array(100, dtype=int32, weak_type=True))


In [3]:
import jax
import jax.numpy as jnp
from flax import struct

from hcnn.solver.solver_sqp import (
    SQPParameters,
    SQPProblemBase,
    sqp_solve,
)


@struct.dataclass
class SimpleProblem(SQPProblemBase[jax.Array]):
    def get_x0(self, phi):
        return jnp.zeros_like(self.cost_quad_diag)

    def constraint_fn_phi(self, x: jax.Array, phi: jax.Array) -> jax.Array:
        return phi * x

    @staticmethod
    def factory():
        return SimpleProblem(
            cost_quad_diag=jnp.array([1.0, 1.0]),
            cost_linear=jnp.array([0, 0]),
            phi=jnp.array([1.0, 1.0]),
            lower_bound=jnp.array([0.5, 0.5]),
            upper_bound=jnp.array([1.0, 1.0]),
        )


osqp_variables = sqp_solve(
    SimpleProblem.factory(),
    SQPParameters(max_iter_lin=1, max_iter_nonlin=1),
)

print(osqp_variables)

(Array([0.49980682, 0.49980682], dtype=float32), SQPInfo(primal_residual=Array([0.00019318, 0.00019318], dtype=float32), dual_residual=Array([1.4901161e-07, 1.4901161e-07], dtype=float32), iterations_lin=Array(50., dtype=float32, weak_type=True), iterations_nonlin=Array(1, dtype=int32, weak_type=True), rho_base_mult=Array(2., dtype=float32), converged_flag=Array(False, dtype=bool)))


In [ ]:
from functools import partial

import jax
from jax import numpy as jnp
from jax.test_util import check_vjp

phi_test = jnp.array([1.0, 1.0])


# Define a simple test function that uses sqp_solve
def test_function(phi):
    problem = SimpleProblem.factory()
    result, info = sqp_solve(
        problem.replace(phi=phi),
        SQPParameters(),
    )
    return result


check_vjp(
    f=test_function,
    f_vjp=partial(jax.vjp, test_function),
    args=(phi_test,),
    rtol=5 / 100,
)

KeyboardInterrupt: 

: 

In [ ]:
@struct.dataclass
class QuadProblem(SQPProblemBase[jax.Array]):
    def get_x0(self, phi):
        return jnp.array([2.0, 2.0])

    def constraint_fn_phi(self, x: jax.Array, phi: jax.Array) -> jax.Array:
        return jnp.concatenate(
            [x[:2], jnp.array([phi[0] * x[0] - phi[1] * x[1] ** 2])]
        )

    @staticmethod
    def factory():
        return QuadProblem(
            cost_quad_diag=jnp.array([0.0, 0.0]),
            cost_linear=jnp.array([0.0, -5]),
            phi=jnp.array([1.0, 1.0]),
            lower_bound=jnp.array([1.0, 1.0, 0.0]),
            upper_bound=jnp.array([2.0, 2.0, 0.0]),
        )


osqp_variables = sqp_solve(
    QuadProblem.factory(),
    SQPParameters(),
)

print(osqp_variables)

(Array([2.0000005, 1.4142138], dtype=float32), SQPInfo(primal_residual=Array([4.7683716e-07, 0.0000000e+00, 2.3841858e-07], dtype=float32), dual_residual=Array([2.0000004e-12, 4.7683716e-07], dtype=float32), iterations_lin=Array(1587.5, dtype=float32, weak_type=True), iterations_nonlin=Array(4, dtype=int32, weak_type=True), rho_base_mult=Array(2.75, dtype=float32), converged_flag=Array(True, dtype=bool)))


In [ ]:
from functools import partial

import jax
from jax import numpy as jnp
from jax.test_util import check_vjp

phi_test = jnp.array([1.0, 1.0])


def test_function(phi):
    problem = QuadProblem.factory()
    x, info = sqp_solve(
        problem.replace(phi=phi),
        SQPParameters(),
    )
    return x  # primal solution only


check_vjp(
    f=test_function,
    f_vjp=partial(jax.vjp, test_function),
    args=(phi_test,),
)